# Iron March — 01: Carga y Limpieza

Pipeline de normalización del dump SQL.
**Entrada:** `IronMarch_2019.11.zip` (IPS 4.x)
**Salida:** parquets limpios en `results/ironmarch/`


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_forum, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)
IM_ZIP = DATA_DIR / 'Far Right Forum' / 'IronMarch_2019.11.zip'
print(f"ZIP exists: {IM_ZIP.exists()}")
print(f"Results dir: {IM_RESULTS}")


## Sección 1 — Carga raw

Cargamos el dump directamente desde el ZIP usando el parser IPS 4.x.
El parser extrae cinco tablas: posts, usuarios, hilos, foros y mensajes privados.


In [ ]:
tables = load_forum(IM_ZIP)

posts_raw   = tables.get('post', pd.DataFrame())
users_raw   = tables.get('user', pd.DataFrame())
threads_raw = tables.get('thread', pd.DataFrame())
forums      = tables.get('forum', pd.DataFrame())
pms_raw     = tables.get('private_message', pd.DataFrame())

print("=== Shapes raw ===")
print(f"  posts:    {posts_raw.shape}")
print(f"  users:    {users_raw.shape}")
print(f"  threads:  {threads_raw.shape}")
print(f"  forums:   {forums.shape}")
print(f"  pms:      {pms_raw.shape}")

print("\n=== Posts columns ===")
print(posts_raw.dtypes)
print("\n=== Users columns ===")
print(users_raw.dtypes)
print("\n=== Date range (posts) ===")
valid_dates = posts_raw['dateline'].dropna() if 'dateline' in posts_raw.columns else pd.Series(dtype='object')
if len(valid_dates):
    print(f"  min: {valid_dates.min()}")
    print(f"  max: {valid_dates.max()}")


## Sección 2 — Pipeline de limpieza

Los pasos de limpieza aplicados son:

1. **Posts:** eliminar filas con `pagetext` nulo o vacío, deduplicar por `postid`, descartar textos menores a 10 caracteres, y asegurar que `dateline` no sea `None`.
2. **Usuarios:** deduplicar por `userid`, descartar filas sin `username`, normalizar el nombre a minúsculas (guardando el original en `username_raw`).
3. **Hilos:** deduplicar por `threadid`.
4. **Foros:** se mantienen sin cambios (tabla de referencia pequeña).
5. **Mensajes privados:** eliminar filas sin `text` o con `dateline` nulo.


In [ ]:
n_raw_posts = len(posts_raw)

posts_clean = posts_raw.copy()

# 1. Drop null / empty pagetext
posts_clean = posts_clean[posts_clean['pagetext'].notna()]
posts_clean = posts_clean[posts_clean['pagetext'].str.strip() != '']

# 2. Drop duplicate postid
posts_clean = posts_clean.drop_duplicates(subset='postid')

# 3. Drop rows with None dateline (parser already filters ~1970, but double-check)
posts_clean = posts_clean[posts_clean['dateline'].notna()]

# 4. Filter texts shorter than 10 characters
posts_clean = posts_clean[posts_clean['pagetext'].str.len() >= 10]

# 5. Ensure userid is string
posts_clean['userid'] = posts_clean['userid'].astype(str)

posts_clean = posts_clean.reset_index(drop=True)

n_clean_posts = len(posts_clean)
print(f"Posts  raw:   {n_raw_posts:,}")
print(f"Posts clean:  {n_clean_posts:,}  (eliminadas: {n_raw_posts - n_clean_posts:,})")


In [ ]:
n_raw_users = len(users_raw)

users_clean = users_raw.copy()

# 1. Drop duplicate userid
users_clean = users_clean.drop_duplicates(subset='userid')

# 2. Drop rows where username is null
users_clean = users_clean[users_clean['username'].notna()]

# 3. Keep original username, add normalized lowercase version
users_clean = users_clean.rename(columns={'username': 'username_raw'})
users_clean['username'] = users_clean['username_raw'].str.strip().str.lower()

users_clean = users_clean.reset_index(drop=True)

n_clean_users = len(users_clean)
print(f"Users  raw:   {n_raw_users:,}")
print(f"Users clean:  {n_clean_users:,}  (eliminadas: {n_raw_users - n_clean_users:,})")


In [ ]:
n_raw_threads = len(threads_raw)

threads_clean = threads_raw.drop_duplicates(subset='threadid').reset_index(drop=True)

n_clean_threads = len(threads_clean)
print(f"Threads  raw:   {n_raw_threads:,}")
print(f"Threads clean:  {n_clean_threads:,}  (eliminadas: {n_raw_threads - n_clean_threads:,})")


In [ ]:
n_raw_pms = len(pms_raw)

pms_clean = pms_raw.copy()

# 1. Drop null or empty text
pms_clean = pms_clean[pms_clean['text'].notna()]
pms_clean = pms_clean[pms_clean['text'].str.strip() != '']

# 2. Drop None dateline
pms_clean = pms_clean[pms_clean['dateline'].notna()]

pms_clean = pms_clean.reset_index(drop=True)

n_clean_pms = len(pms_clean)
print(f"PMs  raw:   {n_raw_pms:,}")
print(f"PMs clean:  {n_clean_pms:,}  (eliminadas: {n_raw_pms - n_clean_pms:,})")


## Sección 3 — Estadísticas post-limpieza

Resumen comparativo de filas antes y después de la limpieza,
y visualización de la actividad de posts por día (raw vs. limpio).


In [ ]:
# Summary table
summary = pd.DataFrame([
    {'Tabla': 'posts',   'Filas raw': n_raw_posts,   'Filas limpias': n_clean_posts,   'Eliminadas': n_raw_posts   - n_clean_posts},
    {'Tabla': 'users',   'Filas raw': n_raw_users,   'Filas limpias': n_clean_users,   'Eliminadas': n_raw_users   - n_clean_users},
    {'Tabla': 'threads', 'Filas raw': n_raw_threads, 'Filas limpias': n_clean_threads, 'Eliminadas': n_raw_threads - n_clean_threads},
    {'Tabla': 'forums',  'Filas raw': len(forums),   'Filas limpias': len(forums),     'Eliminadas': 0},
    {'Tabla': 'pms',     'Filas raw': n_raw_pms,     'Filas limpias': n_clean_pms,     'Eliminadas': n_raw_pms     - n_clean_pms},
])
print(summary.to_string(index=False))

# Posts per day — before vs after
fig, ax = plt.subplots(figsize=(14, 4))

raw_daily = (
    posts_raw['dateline'].dropna()
    .dt.floor('D')
    .value_counts()
    .sort_index()
)
clean_daily = (
    posts_clean['dateline']
    .dt.floor('D')
    .value_counts()
    .sort_index()
)

ax.fill_between(raw_daily.index, raw_daily.values, alpha=0.35, label='raw', color='#8888ff')
ax.fill_between(clean_daily.index, clean_daily.values, alpha=0.65, label='clean', color='#44aaff')
ax.set_title('Posts por día — raw vs. limpio', fontsize=13)
ax.set_xlabel('Fecha')
ax.set_ylabel('Posts')
ax.legend()
plt.tight_layout()
plt.show()


## Sección 4 — Guardar parquets

Se exportan las cinco tablas limpias a `results/ironmarch/`.
Los notebooks 02 y 03 cargarán directamente desde estos parquets.


In [ ]:
for name, df in [
    ('posts',   posts_clean),
    ('users',   users_clean),
    ('threads', threads_clean),
    ('forums',  forums),
    ('pms',     pms_clean),
]:
    path = IM_RESULTS / f'{name}_clean.parquet'
    df.to_parquet(path, index=False)
    print(f"  {name}_clean.parquet: {len(df):,} rows, {path.stat().st_size / 1024:.0f} KB")

print("\nTodos los parquets guardados en:", IM_RESULTS)


## Sección 5 — Validación

Recargamos los parquets guardados y verificamos integridad:
conteos, nulos en columnas clave, y rango de fechas válido.


In [ ]:
print("=== Validación de parquets ===\n")

checks_ok = True

for name, df_original in [
    ('posts',   posts_clean),
    ('users',   users_clean),
    ('threads', threads_clean),
    ('forums',  forums),
    ('pms',     pms_clean),
]:
    path = IM_RESULTS / f'{name}_clean.parquet'
    df_loaded = pd.read_parquet(path)

    row_match = len(df_loaded) == len(df_original)
    print(f"[{name}]")
    print(f"  Filas originales: {len(df_original):,}  |  Recargadas: {len(df_loaded):,}  |  {'✓' if row_match else '✗ MISMATCH'}")

    if not row_match:
        checks_ok = False

# Check no null postid
posts_v = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')
null_postid = posts_v['postid'].isna().sum()
print(f"\n[posts] null postid: {null_postid}  {'✓' if null_postid == 0 else '✗'}")
if null_postid > 0:
    checks_ok = False

# Check no null userid / username
users_v = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')
null_userid   = users_v['userid'].isna().sum()
null_username = users_v['username'].isna().sum()
print(f"[users] null userid: {null_userid}  {'✓' if null_userid == 0 else '✗'}")
print(f"[users] null username: {null_username}  {'✓' if null_username == 0 else '✗'}")
if null_userid > 0 or null_username > 0:
    checks_ok = False

# Check date range — no 1970 dates
min_date = posts_v['dateline'].min()
max_date = posts_v['dateline'].max()
date_ok = min_date.year >= 2000
print(f"\n[posts] fecha mínima: {min_date}  {'✓' if date_ok else '✗ 1970 DATES FOUND'}")
print(f"[posts] fecha máxima: {max_date}")
if not date_ok:
    checks_ok = False

print(f"\n{'=== Validación PASSED ===' if checks_ok else '=== VALIDACIÓN FALLIDA — revisar arriba ==='}")
